# 01 – Downloading SPX option quotes

This notebook downloads one snapshot of all listed S&P 500 index options (SPX and SPXW) from Yahoo
Finance with the `yfinance` package, and saves it as a CSV file in the `data` folder.

The file is saved only when the market is open and the quotes have a bid and an ask, because quotes
taken outside trading hours are not reliable.

**Note.** The rest of the project (notebooks 02–05) uses the snapshot taken on 26 October 2026 at
10:30 New York time, `SPX_2026-10-26_1030.csv`. Running this notebook again creates a new file with a
different name, so the analysis can be repeated on another day by changing `SNAPSHOT` in the next
notebooks.

In [ ]:
import os
import pandas as pd
import yfinance as yf
from datetime import datetime
from zoneinfo import ZoneInfo

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

snapshot_time = datetime.now(ZoneInfo("America/New_York"))   # time of the snapshot, New York time
print(snapshot_time)

## 1. Available expiries

In [ ]:
spx = yf.Ticker("^SPX")
expiries = spx.options          # all listed expiry dates
print(len(expiries), "expiries")
print(expiries)

## 2. Download all option chains

For every expiry we download the calls and the puts, add the option type and the expiry date, and
put everything in one table.

In [ ]:
frames = []

for exp in expiries:
    chain = spx.option_chain(exp)

    calls = chain.calls.copy()
    calls["type"] = "C"
    calls["expiry"] = exp

    puts = chain.puts.copy()
    puts["type"] = "P"
    puts["expiry"] = exp

    frames.append(calls)
    frames.append(puts)

data = pd.concat(frames, ignore_index=True)
print(data.shape)
print(data.head())

## 3. Save the snapshot

The spot price is read from the last downloaded chain. It is used only as a check, because the
forward price is later estimated from put-call parity.

In [ ]:
data["snapshot_time"] = snapshot_time
data["spot"] = chain.underlying["regularMarketPrice"]

market_state = chain.underlying["marketState"]
quoted = ((data["bid"] > 0) & (data["ask"] > 0)).sum()

print("Market state:", market_state)
print("Options with bid and ask:", quoted, "out of", len(data))

if market_state == "REGULAR" and quoted > 0:
    name = "SPX_" + snapshot_time.strftime("%Y-%m-%d_%H%M") + ".csv"
    path = os.path.join(DATA_DIR, name)
    data.to_csv(path, index=False)
    print("Saved:", path)
else:
    print("Market not open or no quotes: file not saved")